In [5]:
import csv
import json
import pandas as pd
import numpy as np
import time
import requests

# Task 1.1: Read appointments.csv using the csv module

In [8]:
print("--- Task 1.1: Read appointments.csv using the csv module ---")
file_path = "/content/appointments.csv"
with open(file_path, mode="r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

def parse_row(row):
    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])
    return row

accepted, rejected = [], []
for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

print(f"[Task 1.1] Total: {len(rows)}, Accepted: {len(accepted)}, Rejected: {len(rejected)}")
with open("rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2)


--- Task 1.1: Read appointments.csv using the csv module ---
[Task 1.1] Total: 1020, Accepted: 988, Rejected: 32


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
# Save rejected rows to JSON for auditing
with open("rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2)

# Task 1.2: Inspect and clean, pass 1

In [11]:
print("--- Task 1.2: Inspect and clean, pass 1  ---")
df = pd.read_csv("/content/appointments.csv")

# Print initial inspection metrics
print("Initial shape:", df.shape)
print("\nInfo:")
df.info()
print("\nNull counts:\n", df.isnull().sum())
print("\nDuplicate count:", df.duplicated().sum())

# (a) Drop exact duplicate rows
df = df.drop_duplicates()
print("\nShape after dropping duplicates:", df.shape)

# (b) Clean booking_channel
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Distinct booking channels after cleaning:", df["booking_channel"].unique())

# (c) Convert date columns
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"], errors="coerce")

# (d) Drop rows where appt_date or status is missing
df = df.dropna(subset=["appt_date", "status"])
print("Final shape after dropping missing dates/statuses:", df.shape)

--- Task 1.2: Inspect and clean, pass 1  ---
Initial shape: (1020, 14)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Null counts:


# Task 1.3 — Clean, pass 2: validity, gaps and outliers

In [12]:
print("--- Task 1.3: Validity Checks, Median Imputation, and Outlier Capping ---")

# Explicitly coerce columns to numeric to ensure clean comparisons
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["distance_km"] = pd.to_numeric(df["distance_km"], errors="coerce")
df["fee_inr"] = pd.to_numeric(df["fee_inr"], errors="coerce")

# 1. Handle invalid ages (anything below 1 or above 100 becomes NaN)
invalid_age_mask = (df["age"] < 1) | (df["age"] > 100)
num_invalid_ages = invalid_age_mask.sum()
df.loc[invalid_age_mask, "age"] = np.nan
print(f"[Check] Found and cleared {num_invalid_ages} invalid age entries.")

# 2. Impute missing values with medians
age_median = df["age"].median()
distance_median = df["distance_km"].median()

df["age"] = df["age"].fillna(age_median).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(distance_median)

print(f"[Imputation] Median age applied: {age_median}")
print(f"[Imputation] Median distance applied: {distance_median}")

# 3. Compute IQR fences and cap outliers for distance and fee
def compute_iqr_caps(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    return lower_fence, upper_fence

dist_lower, dist_upper = compute_iqr_caps(df["distance_km"])
fee_lower, fee_upper = compute_iqr_caps(df["fee_inr"])

dist_capped_count = ((df["distance_km"] > dist_upper) | (df["distance_km"] < dist_lower)).sum()
fee_capped_count = ((df["fee_inr"] > fee_upper) | (df["fee_inr"] < fee_lower)).sum()

df["distance_km"] = np.clip(df["distance_km"], dist_lower, dist_upper)
df["fee_inr"] = np.clip(df["fee_inr"], fee_lower, fee_upper)

# 4. Final rounding and formatting conventions
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print(f"[Outliers] Distance values capped: {dist_capped_count}")
print(f"[Outliers] Fee values capped (keying errors): {fee_capped_count}")
print(f"[Summary] Maximum fee after outlier clipping: {df['fee_inr'].max()}")

--- Task 1.3: Validity Checks, Median Imputation, and Outlier Capping ---
[Check] Found and cleared 5 invalid age entries.
[Imputation] Median age applied: 48.0
[Imputation] Median distance applied: 8.25
[Outliers] Distance values capped: 26
[Outliers] Fee values capped (keying errors): 3
[Summary] Maximum fee after outlier clipping: 1219


#Task 1.4 — Features and a NumPy summary

In [13]:
print("--- Task 1.4: Feature Engineering and NumPy Summaries ---")

# 1. Feature Engineering
# Calculate lead time in days between booking date and appointment date
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

# Create binary target variable: 1 if No-Show, 0 otherwise
df["no_show"] = (df["status"] == "No-Show").astype(int)

# 2. Extract NumPy matrix for calculations
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# (a) Column means (axis=0 means column-wise)
col_means = np.mean(X, axis=0)
print(f"[Means] [age, distance_km, fee_inr, lead_days]: {np.round(col_means, 2)}")

# (b) Lead days analysis (> 14 days) using boolean masks
lead_mask = df["lead_days"] > 14
num_long_lead = lead_mask.sum()
share_long_lead = lead_mask.mean() * 100

print(f"[Lead Time] Rows with lead_days > 14: {num_long_lead} ({share_long_lead:.1f}%)")

# (c) Distance Z-scores using NumPy (population std via np.std)
dist_col = df["distance_km"].to_numpy()
dist_z_scores = (dist_col - np.mean(dist_col)) / np.std(dist_col)
num_extreme_distance = np.sum(np.abs(dist_z_scores) > 2)

print(f"[Z-Scores] Rows with |z| > 2 for distance: {num_extreme_distance}")

# Quick validation of total no-shows
total_noshows = df["no_show"].sum()
noshow_pct = (total_noshows / len(df)) * 100
print(f"[Summary] Total No-Shows: {total_noshows} out of {len(df)} ({noshow_pct:.1f}%)")
print("--- Task 1.4 Complete ---")

--- Task 1.4: Feature Engineering and NumPy Summaries ---
[Means] [age, distance_km, fee_inr, lead_days]: [ 48.17   9.5  635.71  11.82]
[Lead Time] Rows with lead_days > 14: 296 (29.9%)
[Z-Scores] Rows with |z| > 2 for distance: 53
[Summary] Total No-Shows: 218 out of 990 (22.0%)
--- Task 1.4 Complete ---


#Task 1.5 — Enrich the clinics from a live API

In [14]:
print("--- Task 1.5: Enrich Clinics from Live API ---")

# Load the original clinics.csv master file
clinics_df = pd.read_csv("/content/clinics.csv")

temperatures = []

for index, row in clinics_df.iterrows():
    clinic_name = row["clinic_name"]
    lat = row["latitude"]
    lon = row["longitude"]

    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m",
        "forecast_days": 1
    }

    temp_c = None
    success = False

    # Retry loop (up to 2 attempts) for robustness against network issues
    for attempt in range(2):
        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code == 200:
                data = response.json()
                # Take the first value of hourly temperature_2m (midnight UTC reading)
                temp_c = data["hourly"]["temperature_2m"][0]
                success = True
                break
            else:
                print(f"[Warning] {clinic_name}: Status code {response.status_code} on attempt {attempt+1}")
        except requests.exceptions.RequestException as e:
            print(f"[Error] {clinic_name}: Network exception on attempt {attempt+1}: {e}")

        # Brief pause before retrying
        time.sleep(1)

    if not success:
        print(f"[Failed] Could not retrieve temperature for {clinic_name}. Setting as None.")

    temperatures.append(temp_c)

# Add temp_c to the dataframe
clinics_df["temp_c"] = temperatures

# Export to CSV
clinics_enriched_path = "clinics_enriched.csv"
clinics_df.to_csv(clinics_enriched_path, index=False)

print(f"[Success] Exported enriched clinics to {clinics_enriched_path}")
print(clinics_df[["clinic_name", "city", "temp_c"]])

--- Task 1.5: Enrich Clinics from Live API ---
[Success] Exported enriched clinics to clinics_enriched.csv
                clinic_name     city  temp_c
0     Kothrud Family Clinic     Pune    19.4
1      Hadapsar Care Centre     Pune    19.7
2    Sitabuldi Health Point   Nagpur    19.7
3      Malviya Nagar Clinic   Jaipur    16.4
4     Vaishali Nagar Clinic   Jaipur    16.4
5  Edappally Medical Centre    Kochi    26.3
6        Vijay Nagar Clinic   Indore    16.1
7    Gomti Nagar Care Point  Lucknow    14.7


#Task 1.6 — Export the clean table and write the SQL load script

In [15]:
print("--- Task 1.6: Export Clean Table and Generate SQL Script ---")

# 0. Ensure all missing ages and distances are imputed
if df["age"].isnull().sum() > 0:
    df["age"] = pd.to_numeric(df["age"], errors="coerce")
    df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)

if df["distance_km"].isnull().sum() > 0:
    df["distance_km"] = pd.to_numeric(df["distance_km"], errors="coerce")
    df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())

# Ensure lead_days and no_show are present
if "lead_days" not in df.columns:
    df["lead_days"] = (pd.to_datetime(df["appt_date"]) - pd.to_datetime(df["booked_on"])).dt.days
if "no_show" not in df.columns:
    df["no_show"] = (df["status"] == "No-Show").astype(int)

# 1. Sort by appointment_id and format date columns as YYYY-MM-DD
df = df.sort_values("appointment_id").reset_index(drop=True)
df["booked_on"] = pd.to_datetime(df["booked_on"]).dt.strftime("%Y-%m-%d")
df["appt_date"] = pd.to_datetime(df["appt_date"]).dt.strftime("%Y-%m-%d")

# Define the exact 16 required columns in order
columns_order = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status",
    "lead_days", "no_show"
]
df_clean = df[columns_order]

# 2. Pre-export checks[cite: 1]
assert df_clean["appointment_id"].is_unique, "Error: appointment_id is not unique!"
missing_counts = df_clean.isnull().sum().sum()
print(f"[Check] Total missing values across all 16 columns: {missing_counts}")
print(f"[Check] Total rows to export: {len(df_clean)}")
assert missing_counts == 0, "Error: Missing values found in clean dataset!"

# 3. Export appointments_clean.csv
appointments_clean_path = "appointments_clean.csv"
df_clean.to_csv(appointments_clean_path, index=False)
print(f"[Export] Successfully saved {appointments_clean_path}")

# 4. Generate load_data.sql[cite: 1]
clinics_df = pd.read_csv("clinics_enriched.csv")
sql_statements = []

# Write all clinics statements
for _, row in clinics_df.iterrows():
    temp_val = f"{row['temp_c']}" if pd.notna(row['temp_c']) else "NULL"
    stmt = (
        f"INSERT INTO clinics VALUES ("
        f"'{row['clinic_id']}', '{row['clinic_name']}', '{row['city']}', "
        f"'{row['tier']}', {row['latitude']}, {row['longitude']}, {temp_val});"
    )
    sql_statements.append(stmt)

# Write appointments statements[cite: 1]
for _, row in df_clean.iterrows():
    stmt = (
        f"INSERT INTO appointments VALUES ("
        f"'{row['appointment_id']}', '{row['patient_id']}', '{row['clinic_id']}', "
        f"'{row['booked_on']}', '{row['appt_date']}', {row['age']}, "
        f"'{row['gender']}', {row['distance_km']}, {row['fee_inr']}, "
        f"'{row['booking_channel']}', '{row['visit_type']}', '{row['reminder_sent']}', "
        f"{row['prior_no_shows']}, '{row['status']}', {row['lead_days']}, {row['no_show']});"
    )
    sql_statements.append(stmt)

# Save to load_data.sql[cite: 1]
sql_file_path = "load_data.sql"
with open(sql_file_path, "w", encoding="utf-8") as f:
    f.write("\n".join(sql_statements) + "\n")

print(f"[Export] Successfully generated {sql_file_path} with {len(sql_statements)} total statements[cite: 1].")

--- Task 1.6: Export Clean Table and Generate SQL Script ---
[Check] Total missing values across all 16 columns: 0
[Check] Total rows to export: 990
[Export] Successfully saved appointments_clean.csv
[Export] Successfully generated load_data.sql with 998 total statements[cite: 1].
